# DSAR × Lakeflow Declarative Pipelines · 01b · CDC variant (SCD1 silver)

Same as `01_sdp_pipeline`, **except silver is an SCD type-1 dimension** maintained
by **AUTO CDC** (`dp.create_auto_cdc_flow`) keyed on `user_id`. This faithfully
matches Allegiant's real Merlot pipeline (`dbo_user_silver_cdc`) — the exact flow
whose fatal `append-only source` failure started this whole effort.

**Attach as a *separate* Lakeflow pipeline** with its **own target schema** (config
`dsar.schema` = e.g. `allegiant_air_sdp_dsar_cdc`) so it doesn't collide with the
clean-silver variant. Modern API throughout (`from pyspark import pipelines as dp`).

```
raw_user ─stream─▶ bronze_user ─stream─▶ (bronze_user_cdc_feed) ═AUTO CDC═▶ silver_user ─batch MV─▶ gold_user
          mask PII                       change feed for CDC        SCD type 1        aggregate
```

> The `bronze_user_cdc_feed` view is **required** by AUTO CDC (it needs a
> streaming change source), not a stray artifact. `skipChangeCommits` is set on
> every streaming read so an erasure never breaks the pipeline.

## 0. Config

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

def cfg(key, default):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

CATALOG = cfg("dsar.catalog", "dkushari_uc")
SCHEMA  = cfg("dsar.schema",  "allegiant_air_sdp_dsar_cdc")   # own schema
FQ      = f"{CATALOG}.{SCHEMA}"
SOURCE  = f"{FQ}.raw_user"
print("CDC pipeline reads landing source:", SOURCE)

## 1. Bronze — mask PII inline (streaming)

Identical to the primary variant.

In [ ]:
REDACT = "***REDACTED***"

def _mask_json(col):
    # in-JSON masking, native SQL — quote-anchored keys so "name" != "appName"
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{REDACT}\"')"
    return e

@dp.table(
    name="bronze_user",
    comment="Streaming bronze: PII masked inline; user_id/revenue preserved.",
    table_properties={"quality": "bronze"},
)
def bronze_user():
    src = (spark.readStream
           .option("skipChangeCommits", "true")   # survive erasure on raw_user
           .table(SOURCE))
    return src.select(
        "event_id", "user_id",
        F.lit(REDACT).alias("email"),
        F.lit(REDACT).alias("full_name"),
        F.expr(_mask_json("profile_json")).alias("profile_json"),
        "revenue", "event_ts", "_ingest_ts",
    )

## 2. Silver — SCD type 1 via AUTO CDC

`dp.create_streaming_table` declares the target; a streaming **change feed** view
over bronze provides the CDC source; `dp.create_auto_cdc_flow` applies it as SCD
type 1 (one row per `user_id`, latest wins by `_ingest_ts`).

`skipChangeCommits` on the change-feed read of bronze is what keeps this flow from
failing when an erasure deletes from bronze — the fix for the original incident.

In [ ]:
dp.create_streaming_table(
    name="silver_user",
    comment="SCD type-1 customer dimension via AUTO CDC, keyed on user_id.",
    table_properties={"quality": "silver"},
)

@dp.temporary_view(name="bronze_user_cdc_feed")
def bronze_user_cdc_feed():
    # streaming change source for AUTO CDC; skipChangeCommits => an erasure DELETE
    # on bronze is skipped here (02 deletes silver explicitly), keeping the flow alive.
    return (spark.readStream
            .option("skipChangeCommits", "true")
            .table(f"{FQ}.bronze_user"))

dp.create_auto_cdc_flow(
    target="silver_user",
    source="bronze_user_cdc_feed",
    keys=["user_id"],
    sequence_by=F.col("_ingest_ts"),
    stored_as_scd_type=1,
)

## 3. Gold — per-customer aggregate (materialized view)

Reads the SCD1 **silver** dimension. Batch recompute, so erasures propagate on
refresh; you cannot DELETE from it (see `02`).

In [ ]:
@dp.materialized_view(
    name="gold_user",
    comment="Per-customer rollup over the SCD1 silver dimension, recomputed each refresh.",
    table_properties={"quality": "gold"},
)
def gold_user():
    return (spark.read.table(f"{FQ}.silver_user")
            .groupBy("user_id")
            .agg(F.sum("revenue").alias("lifetime_revenue"),
                 F.count("*").alias("event_count"),
                 F.max("event_ts").alias("last_event_ts")))